# Notebook 24. Objective-Regime Manual Verification Atlas

This notebook is the manual-review follow-on to `Notebooks 22–23`.

It is designed to help review events or spells with the full context Scott asked for:

- objective regime label
- spell/episode ID and timing context
- whether offshore JPCZ precedes coastal development
- `925 hPa` divergence map
- `850 hPa q × (-omega)` moisture-proxy map
- existing convergence quicklook / OLR / MODIS satellite panels when available

Use this notebook to review representative events first, then expand toward the full ~200-event manual pass.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = os.environ.get("JPCZ_CATALOG_BRANCH", "codex/notebook16-pcolormesh")
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

def clone_repo_branch():
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )

def sync_repo_branch():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)


if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    print("Using existing repo clone:", REPO_DIR)

try:
    sync_repo_branch()
except subprocess.CalledProcessError:
    print("Existing clone could not switch branches cleanly. Re-cloning target branch.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    clone_repo_branch()
    sync_repo_branch()

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

active_branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], text=True, capture_output=True, check=True).stdout.strip()
print("Working directory:", os.getcwd())
print("Runtime repo branch:", active_branch)


In [ ]:
from pathlib import Path
import shutil

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Image, Markdown, display

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import JPCZ_POLYGON_VERTICES, OBJECTIVE_SUBTYPE_DOMAIN
from jpcz_catalog.detect import prepare_detection_geometry
from jpcz_catalog.objective_regimes import build_objective_manual_review_scaffold
from jpcz_catalog.satellite import default_modis_layers_for_date

OBJECTIVE_EXPORT_DIR = Path("outputs/verification/objective_coastal_box_regimes")
TIMING_EXPORT_DIR = Path("outputs/verification/objective_regime_timing_and_impact")
EOF_EXPORT_DIR = Path("outputs/verification/objective_subtype_eof_analysis")
REVIEW_EXPORT_DIR = Path("outputs/verification/objective_regime_manual_review")
REVIEW_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

OBJECTIVE_EVENT_METRICS_PATH = OBJECTIVE_EXPORT_DIR / "objective_coastal_box_event_metrics.csv"
EVENT_SEQUENCE_PATH = TIMING_EXPORT_DIR / "objective_regime_event_sequences_gap36h.csv"
EPISODE_SUMMARY_PATH = TIMING_EXPORT_DIR / "objective_regime_episode_summary_gap36h.csv"
MANUAL_BASE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_merged_12h_manual_verification.csv")
LOW_LEVEL_STACK_PATH = EOF_EXPORT_DIR / "cleaned_low_level_eof_event_fields.nc"
REVIEW_SCAFFOLD_PATH = REVIEW_EXPORT_DIR / "objective_regime_manual_review_scaffold.csv"

QUICKLOOK_DIR = Path("outputs/verification/ndjf_event_quicklooks_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")

REVIEW_EVENT_INDEX = 0
REVIEW_EPISODE_ID = None
MOISTURE_FIELD = "vertical_moisture_flux_proxy_850_peak"
DIVERGENCE_FIELD = "divergence_925_peak"
COASTAL_STRIP_VERTICES = (
    (133.05, 35.55),
    (136.05, 35.55),
    (139.55, 39.00),
    (139.55, 42.55),
)


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive cache:", drive_path)
    return True


def load_cached_dataset(path: Path):
    if not path.exists():
        return None
    with xr.open_dataset(path) as ds:
        return ds.load()


def quicklook_filename(event_index: int, event_peak: pd.Timestamp) -> str:
    return f"{pd.Timestamp(event_peak).strftime('%Y%m%dT%H%M')}_idx{int(event_index):03d}.png"


def satellite_filename(event_index: int, event_peak: pd.Timestamp) -> str | None:
    layers = default_modis_layers_for_date(pd.Timestamp(event_peak).normalize())
    if not layers:
        return None
    layer_slug = (
        layers[0].replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    timestamp = pd.Timestamp(event_peak).strftime('%Y%m%dT%H%M')
    return f"{timestamp}_idx{int(event_index):03d}_{layer_slug}.jpg"


def first_existing_path(local_dir: Path, filename: str | None) -> Path | None:
    if filename is None:
        return None
    local_path = local_dir / filename
    if local_path.exists():
        return local_path
    if PERSIST_OUTPUTS_TO_DRIVE:
        drive_path = Path(DRIVE_OUTPUT_DIR) / local_dir.name / filename
        if drive_path.exists():
            return drive_path
    return None


In [ ]:
paths_to_restore = [
    OBJECTIVE_EVENT_METRICS_PATH,
    EVENT_SEQUENCE_PATH,
    EPISODE_SUMMARY_PATH,
    MANUAL_BASE_PATH,
    LOW_LEVEL_STACK_PATH,
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

required_paths = {
    "Notebook 22 objective labels": OBJECTIVE_EVENT_METRICS_PATH,
    "Notebook 23 event sequences": EVENT_SEQUENCE_PATH,
    "Notebook 23 episode summary": EPISODE_SUMMARY_PATH,
    "Manual verification base catalog": MANUAL_BASE_PATH,
    "Notebook 17 low-level event stack": LOW_LEVEL_STACK_PATH,
}
missing_paths = [f"{label}: {path}" for label, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise RuntimeError("Missing required inputs:\n- " + "\n- ".join(missing_paths))

objective_df = pd.read_csv(
    OBJECTIVE_EVENT_METRICS_PATH,
    parse_dates=[column for column in ["event_start", "event_end", "event_peak", "event_peak_jst"] if column in pd.read_csv(OBJECTIVE_EVENT_METRICS_PATH, nrows=0).columns],
)
if "event_index" not in objective_df.columns:
    objective_df["event_index"] = objective_df.index.astype(int)

sequence_df = pd.read_csv(
    EVENT_SEQUENCE_PATH,
    parse_dates=[column for column in ["event_start", "event_end", "event_peak", "event_peak_jst", "first_event_peak", "last_event_peak", "first_offshore_peak", "first_coastal_peak"] if column in pd.read_csv(EVENT_SEQUENCE_PATH, nrows=0).columns],
)
if "event_index" not in sequence_df.columns:
    sequence_df["event_index"] = sequence_df.index.astype(int)

episode_summary_df = pd.read_csv(
    EPISODE_SUMMARY_PATH,
    parse_dates=[column for column in ["episode_start", "episode_end", "first_event_peak", "last_event_peak", "first_event_peak_jst", "last_event_peak_jst", "first_offshore_peak", "first_coastal_peak"] if column in pd.read_csv(EPISODE_SUMMARY_PATH, nrows=0).columns],
)
manual_base_df = pd.read_csv(
    MANUAL_BASE_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)
manual_base_df = add_japan_local_time_columns(manual_base_df)
manual_base_df["event_index"] = manual_base_df.index.astype(int)

merge_columns = [
    "event_index",
    "objective_regime_label",
    "objective_episode_id",
    "objective_episode_gap_hours",
    "objective_episode_regime_path",
    "offshore_precedes_coastal",
    "offshore_to_coastal_lag_hours",
    "event_peak_jst",
]
review_df = manual_base_df.merge(
    sequence_df[[column for column in merge_columns if column in sequence_df.columns]],
    on="event_index",
    how="left",
    suffixes=("", "_sequence"),
)
review_df = build_objective_manual_review_scaffold(review_df)
review_df.to_csv(REVIEW_SCAFFOLD_PATH, index=False)
maybe_copy_to_drive(REVIEW_SCAFFOLD_PATH)

low_level_stack_ds = load_cached_dataset(LOW_LEVEL_STACK_PATH)
if low_level_stack_ds is None:
    raise RuntimeError("Unable to restore the low-level event stack from Notebook 17.")

if REVIEW_EPISODE_ID:
    episode_rows_df = review_df.loc[review_df["objective_episode_id"] == REVIEW_EPISODE_ID].copy()
    if episode_rows_df.empty:
        raise RuntimeError(f"No review rows were found for REVIEW_EPISODE_ID={REVIEW_EPISODE_ID!r}")
    selected_row = episode_rows_df.sort_values("event_peak").iloc[0].copy()
else:
    if REVIEW_EVENT_INDEX not in review_df["event_index"].values:
        raise RuntimeError(f"REVIEW_EVENT_INDEX={REVIEW_EVENT_INDEX} is not present in the review table.")
    selected_row = review_df.loc[review_df["event_index"] == REVIEW_EVENT_INDEX].iloc[0].copy()
    episode_rows_df = review_df.loc[review_df["objective_episode_id"] == selected_row["objective_episode_id"]].copy()

selected_event_index = int(selected_row["event_index"])
selected_peak = pd.Timestamp(selected_row["event_peak"])
selected_event_ds = low_level_stack_ds.sel(event_index=selected_event_index)
geometry_polygon = prepare_detection_geometry(selected_event_ds.longitude, selected_event_ds.latitude, JPCZ_POLYGON_VERTICES)
geometry_coastal = prepare_detection_geometry(selected_event_ds.longitude, selected_event_ds.latitude, COASTAL_STRIP_VERTICES)

print("Notebook 24 selected review event")
display(review_df.loc[review_df["event_index"] == selected_event_index, [
    "event_index",
    "event_start",
    "event_end",
    "event_peak",
    "event_peak_jst",
    "objective_regime_label",
    "objective_episode_id",
    "objective_episode_regime_path",
    "offshore_precedes_coastal",
    "offshore_to_coastal_lag_hours",
    "verified_event",
    "cloud_band_present",
    "satellite_checked",
    "satellite_cloud_band_match",
]].T)

print("\nEpisode context for the selected review event")
display(episode_rows_df[[
    "event_index",
    "event_peak_jst",
    "objective_regime_label",
    "objective_episode_regime_path",
    "offshore_to_coastal_lag_hours",
]].sort_values("event_peak_jst"))


In [ ]:
required_globals = [
    "selected_event_ds",
    "geometry_polygon",
    "geometry_coastal",
    "selected_row",
    "MOISTURE_FIELD",
    "DIVERGENCE_FIELD",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the Notebook 24 review-setup cell first. Missing globals: {missing_globals}")



def add_region_overlays(ax):
    polygon_vertices = np.array(JPCZ_POLYGON_VERTICES)
    coastal_vertices = np.array(COASTAL_STRIP_VERTICES)
    ax.plot(
        np.r_[polygon_vertices[:, 0], polygon_vertices[0, 0]],
        np.r_[polygon_vertices[:, 1], polygon_vertices[0, 1]],
        color="#1f77b4",
        linewidth=2.2,
        transform=ccrs.PlateCarree(),
        label="JPCZ polygon",
    )
    ax.plot(
        np.r_[coastal_vertices[:, 0], coastal_vertices[0, 0]],
        np.r_[coastal_vertices[:, 1], coastal_vertices[0, 1]],
        color="#d95f02",
        linewidth=2.2,
        transform=ccrs.PlateCarree(),
        label="Coastal wedge",
    )


fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.8),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)

plot_specs = [
    (MOISTURE_FIELD, "850 hPa q×(-ω) proxy at event peak", "BrBG"),
    (DIVERGENCE_FIELD, "925 hPa signed divergence at event peak", "RdBu_r"),
]

for ax, (field_name, title, cmap_name) in zip(axes, plot_specs):
    field = selected_event_ds[field_name]
    values = field.values.astype(float)
    finite_values = values[np.isfinite(values)]
    max_abs = float(np.nanmax(np.abs(finite_values))) if finite_values.size else 1.0
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0
    norm = mcolors.TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs)
    mesh = ax.pcolormesh(
        field.longitude,
        field.latitude,
        field,
        cmap=cmap_name,
        norm=norm,
        shading="auto",
        transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.8)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.4)
    ax.set_extent([
        float(field.longitude.min()),
        float(field.longitude.max()),
        float(field.latitude.min()),
        float(field.latitude.max()),
    ], crs=ccrs.PlateCarree())
    add_region_overlays(ax)
    gl = ax.gridlines(draw_labels=True, linestyle=":", linewidth=0.4, alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title)
    cbar = fig.colorbar(mesh, ax=ax, shrink=0.86, pad=0.02)
    cbar.ax.set_ylabel(field_name)

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=2)
fig.suptitle(
    f"Objective review event {selected_event_index} | {selected_peak:%Y-%m-%d %H:%M UTC} | {selected_row['objective_regime_label']}",
    fontsize=13,
)
plt.show()


In [ ]:
required_globals = [
    "selected_event_index",
    "selected_peak",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the Notebook 24 review-setup cell first. Missing globals: {missing_globals}")

quicklook_path = first_existing_path(QUICKLOOK_DIR, quicklook_filename(selected_event_index, selected_peak))
olr_path = first_existing_path(OLR_DIR, quicklook_filename(selected_event_index, selected_peak))
satellite_path = first_existing_path(SATELLITE_DIR, satellite_filename(selected_event_index, selected_peak))

for title, panel_path in [
    ("Convergence quicklook", quicklook_path),
    ("OLR / cloud-proxy panel", olr_path),
    ("MODIS daily satellite panel", satellite_path),
]:
    if panel_path is None:
        print(f"{title}: not found locally or in Drive cache for this event.")
        continue
    display(Markdown(f"**{title}**  \n`{panel_path}`"))
    display(Image(filename=str(panel_path), width=650))


In [ ]:
required_globals = [
    "selected_row",
    "episode_rows_df",
    "REVIEW_SCAFFOLD_PATH",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(f"Run the Notebook 24 earlier cells before the summary cell. Missing globals: {missing_globals}")

print("Notebook 24 summary")
print("- This notebook is the manual-review atlas for the simplified objective-regime workflow.")
print("- The selected event is shown with its objective label, spell context, moisture proxy, divergence, and any saved quicklook / OLR / MODIS panels.")
print("- The review scaffold now carries both the older manual-verification columns and the new objective-regime timing columns.")
print(f"- Review scaffold: {REVIEW_SCAFFOLD_PATH}")
print("\nSuggested manual checklist for this selected event")
print("- Is the event really offshore JPCZ-core, coastal-interaction, mixed, or weak?")
print("- Is a cloud band present and consistent with the objective label?")
print("- Does the event occur before, during, or after coastal development within its spell?")
print("- Is the coastal impact obvious near the coast/Japan interface, and where?")
print("- Do the satellite, divergence, and moisture-proxy panels tell the same story?")
